# Train Rnai LLM v2 (3 epochs + auto-push)

QLoRA fine-tune of Qwen2.5-7B (Thai/ASEAN + builder track).

**Run order:** T4 GPU → install → **login (do this FIRST)** → upload data → load model → **train+push (auto)**.

The train cell pushes to `naiguitarfolk/rnai-llm` automatically the moment training
finishes — so a long run can't be lost by a disconnect or wrong-session push.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
# Log in NOW so the train cell can auto-push when it finishes (paste a WRITE token)
from huggingface_hub import login
login()

In [ ]:
# (Optional but recommended for long runs) save checkpoints to Google Drive so a
# disconnect doesn't lose progress. Skip if you don't want Drive.
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/rnai-llm-outputs'

In [ ]:
# Upload your training file (pick rnai_all.train.jsonl ~6k rows)
from google.colab import files
up = files.upload()
DATA = next(iter(up))
print("dataset:", DATA)
!wc -l "$DATA"

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048, load_in_4bit=True, dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

In [ ]:
import json
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

rows = [json.loads(l) for l in open(DATA, encoding="utf-8") if l.strip()]
print(f"loaded {len(rows)} examples from {DATA}")
assert rows, "Dataset empty — re-run the upload cell and pick rnai_all.train.jsonl"

_base = Dataset.from_list(rows)
ds = _base.map(
    lambda e: {"text": tokenizer.apply_chat_template(e["messages"], tokenize=False, add_generation_prompt=False)},
    remove_columns=_base.column_names,
)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=2048,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=10, num_train_epochs=3,   # use 2 if T4 time is tight
        learning_rate=2e-4, logging_steps=10, optim="adamw_8bit",
        weight_decay=0.01, lr_scheduler_type="linear", seed=42,
        output_dir="outputs", report_to="none",
    ),
)
trainer.train()

# ---- AUTO-PUSH the moment training finishes (same session = right weights) ----
print("training done — pushing to naiguitarfolk/rnai-llm ...")
model.push_to_hub("rnai-llm")
tokenizer.push_to_hub("rnai-llm")
print("\u2705 pushed: https://huggingface.co/naiguitarfolk/rnai-llm")

## After it pushes — refresh the live server (on your Mac)
```bash
cd ~/rnai-mobile/docs/reference/deploy
python3 -m modal deploy modal_app.py
```
Test:
```bash
curl -s https://naiguitarfolk--rnai-backup-serve.modal.run/v1/chat/completions -H "Content-Type: application/json" -d '{"model":"rnai-llm","messages":[{"role":"system","content":"You are Rnai, the AI assistant of Rnai.io. Reply in the user language."},{"role":"user","content":"คุณคือใคร"}]}'
```